# Multimodal Traffic Counting - Raw Data Ingestion

## Objective

This notebook retrieves multimodal traffic counting data from the Paris OpenData API and stores the raw JSON response in a Databricks Volume.

### Data Flow

Paris OpenData API → Raw JSON → Databricks Volume → Spark DataFrame

### Source

Paris OpenData dataset: Comptage multimodal des déplacements à Paris

### API Endpoint

https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/comptage-multimodal-comptages/exports/json

### Storage Location

/Volumes/workspace/default/raw_data/comptage_multimodal_raw.json

### Filters

The API request retrieves data for the following counting locations:

- CF0181_101 rue d'Amsterdam
- 134 quai de Jemmapes
- CF0004_2 boulevard Montmartre (sens E-O)
- CF0256_88 rue de Rivoli

### Processing

1. Create the Databricks Volume directory if it does not already exist.
2. Send a GET request to the Paris OpenData API.
3. Apply the specified location filters.
4. Download the API response in JSON format.
5. Save the raw JSON response to the Databricks Volume.
6. Read the JSON file using Spark.
7. If the JSON contains a results column, explode the results array to extract individual records.

### Output

Raw data is preserved as:

/Volumes/workspace/default/raw_data/comptage_multimodal_raw.json

The resulting Spark DataFrame contains the individual traffic counting records for further processing and analysis.

In [0]:
import os
import requests

# 1. API URL & Volume Path Setup
BASE_URL = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/comptage-multimodal-comptages/exports/json"
VOLUME_PATH = "/Volumes/workspace/default/raw_data"
FILE_NAME = "comptage_multimodal_raw.json"
TARGET_FILE_PATH = os.path.join(VOLUME_PATH, FILE_NAME)

os.makedirs(VOLUME_PATH, exist_ok=True)

# 2. Filters
params = {
    "refine": [
        'label:"CF0181_101 rue d\'Amsterdam"',
        'label:"134 quai de Jemmapes"',
        'label:"CF0004_2 boulevard Montmartre (sens E-O)"',
        'label:"CF0256_88 rue de Rivoli"'
    ]
}

# 3. Download all records
response = requests.get(BASE_URL, params=params)
response.raise_for_status()

# 4. Save JSON to Volume
with open(TARGET_FILE_PATH, "wb") as f:
    f.write(response.content)

print(f"Successfully saved data to {TARGET_FILE_PATH}!")

In [0]:
# 5. Read JSON with Spark
df = spark.read.option("multiline", "true").json(TARGET_FILE_PATH)

# 6. If the JSON contains "results", extract the records
if "results" in df.columns:
    df = df.selectExpr("explode(results) as data").select("data.*")

In [0]:

# 7. Set the correct column order
column_order = [
    "id_trajectoire",
    "id_site",
    "label",
    "t",
    "mode",
    "nb_usagers",
    "voie",
    "sens",
    "trajectoire",
    "coordonnees_geo"
]

df = df.select(*column_order)

# 8. Display table
display(df)

In [0]:
# 9. Count the number of rows
print("Number of rows:", df.count())